# Speech Emotion Recognition Audio Augmentation

Notebook ini bertujuan untuk memperluas dataset pelatihan menggunakan teknik audio augmentation.

Tahapan yang dilakukan:

1. Membaca seluruh audio
2. Membuat beberapa variasi audio
3. Mengubah seluruh audio menjadi Mel Spectrogram
4. Melakukan padding dan normalisasi
5. Menyimpan dataset hasil augmentasi

In [1]:
# ==========================================================
# IMPORT LIBRARY
# ==========================================================

import os

import librosa

import numpy as np

from tqdm import tqdm

In [2]:
# ==========================================================
# CONFIGURATION
# ==========================================================

DATASET_PATH = "Dataset_Final_Indonesia"

SAMPLE_RATE = 16000

N_MELS = 128

MAX_LENGTH = 215

RANDOM_STATE = 42

In [3]:
# ==========================================================
# EMOTION LABEL
# ==========================================================

emotion_map = {

    "01_Anger": 0,

    "02_Sadness": 1,

    "03_Neutral": 2,

    "04_Happiness": 3

}

In [4]:
# ==========================================================
# CHECK DATASET
# ==========================================================

print("Dataset Folder :")

print(DATASET_PATH)

print("\nDataset Summary\n")

total_audio = 0

for emotion in emotion_map.keys():

    folder = os.path.join(
        DATASET_PATH,
        emotion
    )

    total = len(os.listdir(folder))

    total_audio += total

    print(f"{emotion:<15}: {total}")

print("\nTotal Audio :", total_audio)

Dataset Folder :
Dataset_Final_Indonesia

Dataset Summary

01_Anger       : 400
02_Sadness     : 400
03_Neutral     : 461
04_Happiness   : 460

Total Audio : 1721


In [5]:
# ==========================================================
# LOAD AUDIO
# ==========================================================

def load_audio(audio_path):

    audio, sr = librosa.load(

        audio_path,

        sr=SAMPLE_RATE

    )

    return audio, sr

In [6]:
# ==========================================================
# ADD GAUSSIAN NOISE
# ==========================================================

def add_noise(audio):

    noise = np.random.normal(

        0,

        0.005,

        len(audio)

    )

    return audio + noise

In [7]:
# ==========================================================
# PITCH SHIFT
# ==========================================================

def pitch_shift(audio, sr):

    return librosa.effects.pitch_shift(

        y=audio,

        sr=sr,

        n_steps=2

    )

In [8]:
# ==========================================================
# TIME STRETCH
# ==========================================================

def time_stretch(audio):

    return librosa.effects.time_stretch(

        y=audio,

        rate=0.9

    )

In [9]:
# ==========================================================
# CREATE MEL SPECTROGRAM
# ==========================================================

def create_mel_spectrogram(audio, sr):

    mel = librosa.feature.melspectrogram(

        y=audio,

        sr=sr,

        n_mels=N_MELS

    )

    mel_db = librosa.power_to_db(

        mel,

        ref=np.max

    )

    return mel_db

In [10]:
# ==========================================================
# PAD OR TRIM
# ==========================================================

def pad_or_trim(mel_db):

    pad_width = MAX_LENGTH - mel_db.shape[1]

    if pad_width > 0:

        mel_db = np.pad(

            mel_db,

            ((0, 0), (0, pad_width)),

            mode="constant"

        )

    else:

        mel_db = mel_db[:, :MAX_LENGTH]

    return mel_db

In [11]:
# ==========================================================
# NORMALIZE FEATURE
# ==========================================================

def normalize_feature(feature):

    feature = feature.astype("float32")

    denominator = feature.max() - feature.min()

    if denominator == 0:

        return np.zeros_like(feature)

    feature = (

        feature - feature.min()

    ) / denominator

    return feature

In [12]:
# ==========================================================
# PROCESS FEATURE
# ==========================================================

def process_feature(audio, sr):

    mel_db = create_mel_spectrogram(

        audio,

        sr

    )

    mel_db = pad_or_trim(

        mel_db

    )

    mel_db = normalize_feature(

        mel_db

    )

    return mel_db

In [13]:
# ==========================================================
# SAVE FEATURE
# ==========================================================

def save_feature(feature, label):

    X.append(feature)

    y.append(label)

In [14]:
# ==========================================================
# BUILD AUGMENTED DATASET
# ==========================================================

X = []

y = []

processed_files = 0

failed_files = []

for emotion, label in emotion_map.items():

    folder = os.path.join(
        DATASET_PATH,
        emotion
    )

    print(f"\nProcessing {emotion}...")

    audio_files = sorted(os.listdir(folder))

    for file_name in tqdm(audio_files):

        try:

            audio_path = os.path.join(
                folder,
                file_name
            )

            audio, sr = load_audio(
                audio_path
            )

            # -----------------------------
            # Original
            # -----------------------------

            save_feature(

                process_feature(
                    audio,
                    sr
                ),

                label

            )

            # -----------------------------
            # Noise
            # -----------------------------

            save_feature(

                process_feature(
                    add_noise(audio),
                    sr
                ),

                label

            )

            # -----------------------------
            # Pitch Shift
            # -----------------------------

            save_feature(

                process_feature(
                    pitch_shift(
                        audio,
                        sr
                    ),
                    sr
                ),

                label

            )

            # -----------------------------
            # Time Stretch
            # -----------------------------

            save_feature(

                process_feature(
                    time_stretch(audio),
                    sr
                ),

                label

            )

            processed_files += 1

        except Exception as e:

            failed_files.append(

                (file_name, str(e))

            )


Processing 01_Anger...


100%|████████████████████████████████████████████████████████████████████████████████| 400/400 [00:44<00:00,  9.09it/s]



Processing 02_Sadness...


100%|████████████████████████████████████████████████████████████████████████████████| 400/400 [00:41<00:00,  9.57it/s]



Processing 03_Neutral...


 90%|███████████████████████████████████████████████████████████████████████▊        | 414/461 [00:43<00:06,  7.69it/s]C:\Users\hafiz\AppData\Local\Temp\ipykernel_40604\3440197475.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
C:\Users\hafiz\anaconda3\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
100%|████████████████████████████████████████████████████████████████████████████████| 461/461 [00:50<00:00,  9.12it/s]



Processing 04_Happiness...


100%|████████████████████████████████████████████████████████████████████████████████| 460/460 [00:53<00:00,  8.56it/s]


In [15]:
# ==========================================================
# CONVERT TO NUMPY
# ==========================================================

X = np.array(

    X,

    dtype="float32"

)

X = X[..., np.newaxis]

y = np.array(

    y,

    dtype="int32"

)

print("Augmented dataset successfully created.")

Augmented dataset successfully created.


In [16]:
# ==========================================================
# DATASET INFORMATION
# ==========================================================

print(f"X Shape : {X.shape}")

print(f"y Shape : {y.shape}")

print(f"Processed Files : {processed_files}")

print(f"Failed Files : {len(failed_files)}")

X Shape : (6884, 128, 215, 1)
y Shape : (6884,)
Processed Files : 1721
Failed Files : 0


In [17]:
# ==========================================================
# LABEL DISTRIBUTION
# ==========================================================

label_names = {

    0: "Anger",

    1: "Sadness",

    2: "Neutral",

    3: "Happiness"

}

print("Augmented Dataset Distribution\n")

unique, counts = np.unique(
    y,
    return_counts=True
)

for label, total in zip(unique, counts):

    print(f"{label_names[label]:<12}: {total}")

Augmented Dataset Distribution

Anger       : 1600
Sadness     : 1600
Neutral     : 1844
Happiness   : 1840


In [18]:
# ==========================================================
# SAVE AUGMENTED DATASET
# ==========================================================

np.save(

    "X_dataset_aug.npy",

    X

)

np.save(

    "y_dataset_aug.npy",

    y

)

print("Augmented dataset saved successfully.")

Augmented dataset saved successfully.


In [19]:
# ==========================================================
# VERIFY SAVED FILE
# ==========================================================

print("Saved Files\n")

files = [

    "X_dataset_aug.npy",

    "y_dataset_aug.npy"

]

for file in files:

    if os.path.exists(file):

        size = os.path.getsize(file) / (1024 * 1024)

        print(f"{file:<20} {size:.2f} MB")

    else:

        print(f"{file:<20} Not Found")

Saved Files

X_dataset_aug.npy    722.69 MB
y_dataset_aug.npy    0.03 MB


In [20]:
# ==========================================================
# SUMMARY
# ==========================================================

print("=" * 60)

print("AUDIO AUGMENTATION COMPLETED")

print("=" * 60)

print(f"Original Audio        : {processed_files}")

print(f"Augmented Samples     : {len(y)}")

print(f"Feature Shape         : {X.shape}")

print(f"Label Shape           : {y.shape}")

print(f"Output Feature File   : X_dataset_aug.npy")

print(f"Output Label File     : y_dataset_aug.npy")

print("=" * 60)

AUDIO AUGMENTATION COMPLETED
Original Audio        : 1721
Augmented Samples     : 6884
Feature Shape         : (6884, 128, 215, 1)
Label Shape           : (6884,)
Output Feature File   : X_dataset_aug.npy
Output Label File     : y_dataset_aug.npy
